# Jax-gcm (physics = optional)

In [1]:
import xarray as xr
import numpy as np
import jax.numpy as jnp
import jax_datetime as jdt

from jcm.model import Model
from jcm.physics_interface import PhysicsState, Physics


W0225 10:21:56.474740 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:56.897003 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:56.945156 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


In [2]:
from jcm.geometry import Geometry
from jcm.model import Model
import numpy as np

# Geometry
TERRAIN = "/home/gmathieu/src/jax-gcm/jcm/data/bc/t30/clim/terrain.nc"
geom = Geometry.from_file(TERRAIN, target_resolution=31, num_levels=8)

# Model (sans physics)
m = Model(geometry=geom, physics=None)

# Extraire les axes nodaux horizontaux
lon_rad = np.array(m.coords.horizontal.nodal_axes[0])
lat_rad = np.array(m.coords.horizontal.nodal_axes[1])

lon_deg = lon_rad * 180/np.pi
lat_deg = lat_rad * 180/np.pi

print("lon_deg:", lon_deg.shape, lon_deg.min(), lon_deg.max())
print("lat_deg:", lat_deg.shape, lat_deg.min(), lat_deg.max())


W0225 10:21:57.137523 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.439068 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.584650 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.632091 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.684283 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.733621 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 10:21:57.778987 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. 

lon_deg: (96,) 0.0 356.25
lat_deg: (48,) -57.22536341559415 57.22536341559415


W0225 10:22:00.347383 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


In [3]:
import time
import xarray as xr
import numpy as np

from dinosaur import horizontal_interpolation
from dinosaur import spherical_harmonic
from dinosaur import xarray_utils

ERA5_PATH = "/scratch/gmathieu/era5_snapshots_5days_3h_1990/era5_19900501T00.nc"
ds_era = xr.open_dataset(ERA5_PATH, chunks=None)

lat_name = "latitude" if "latitude" in ds_era.dims else "lat"
lon_name = "longitude" if "longitude" in ds_era.dims else "lon"

era = ds_era

print("Opened ERA5:", ERA5_PATH)
print("dims:", dict(era.sizes))
print("data_vars:", list(era.data_vars)[:20], "..." if len(era.data_vars) > 20 else "")

# --- Normalize lon/lat conventions ---
# lon in 0..360
if float(era[lon_name].min()) < 0:
    era = era.assign_coords({lon_name: (era[lon_name] % 360)}).sortby(lon_name)

# lat increasing
if float(era[lat_name][0]) > float(era[lat_name][-1]):
    era = era.sortby(lat_name)

# If time exists, usually this is a single snapshot; slice to avoid extra dims.
if "time" in era.dims and era.sizes["time"] == 1:
    era = era.isel(time=0)

wanted = [
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "specific_humidity",
    "surface_pressure",
    "geopotential_at_surface",
]

present = [v for v in wanted if v in era.data_vars]
missing = [v for v in wanted if v not in era.data_vars]
print("missing:", missing)
era = era[present]


Opened ERA5: /scratch/gmathieu/era5_snapshots_5days_3h_1990/era5_19900501T00.nc
dims: {'latitude': 721, 'longitude': 1440, 'hybrid': 137}
data_vars: ['surface_pressure', 'geopotential_at_surface', 'temperature', 'u_component_of_wind', 'v_component_of_wind', 'specific_humidity'] 
missing: []


In [4]:
import xarray as xr
import numpy as np
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

# Prendre un snapshot pour inférer la grille source
era0_raw = ds_era.isel(time=0) if "time" in ds_era.dims else ds_era

# normaliser lat/lon
if "latitude" in era0_raw.coords and float(era0_raw["latitude"][0]) > float(era0_raw["latitude"][-1]):
    era0_raw = era0_raw.sortby("latitude")
if "longitude" in era0_raw.coords and float(era0_raw["longitude"].min()) < 0:
    era0_raw = era0_raw.assign_coords(longitude=((era0_raw["longitude"] + 360) % 360)).sortby("longitude")

# cible = grille modèle (1D)
lon_da = xr.DataArray(np.asarray(lon_deg, dtype=np.float32), dims=("longitude",), name="longitude")
lat_da = xr.DataArray(np.asarray(lat_deg, dtype=np.float32), dims=("latitude",),  name="latitude")

src_grid = spherical_harmonic.Grid(
    latitude_nodes=era0_raw.sizes["latitude"],
    longitude_nodes=era0_raw.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(era0_raw["latitude"]),
    longitude_offset=xarray_utils.infer_longitude_offset(era0_raw["longitude"]),
)
tgt_grid = spherical_harmonic.Grid(
    latitude_nodes=lat_da.size,
    longitude_nodes=lon_da.size,
    latitude_spacing=xarray_utils.infer_latitude_spacing(lat_da),
    longitude_offset=xarray_utils.infer_longitude_offset(lon_da),
)

regridder = horizontal_interpolation.BilinearRegridder(src_grid, tgt_grid)
print("Regridder ready:",
      "src", (src_grid.latitude_nodes, src_grid.longitude_nodes),
      "-> tgt", (tgt_grid.latitude_nodes, tgt_grid.longitude_nodes))

Regridder ready: src (721, 1440) -> tgt (48, 96)


In [5]:
import time
import xarray as xr
import numpy as np
from dinosaur import xarray_utils

wanted = [
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "specific_humidity",
    "surface_pressure",
    "geopotential_at_surface",
]

# snapshot t0 (ou dataset unique sans time)
era0 = ds_era.isel(time=0) if "time" in ds_era.dims else ds_era

# normaliser lat/lon
if "latitude" in era0.coords and float(era0["latitude"][0]) > float(era0["latitude"][-1]):
    era0 = era0.sortby("latitude")
if "longitude" in era0.coords and float(era0["longitude"].min()) < 0:
    era0 = era0.assign_coords(longitude=((era0["longitude"] + 360) % 360)).sortby("longitude")

present = [v for v in wanted if v in era0.data_vars]
missing = [v for v in wanted if v not in era0.data_vars]
print("missing:", missing)
era0 = era0[present]

# enlever time singleton si jamais présent
if "time" in era0.dims and era0.sizes["time"] == 1:
    era0 = era0.isel(time=0)

# cast float32
for v in present:
    if np.issubdtype(era0[v].dtype, np.floating) and era0[v].dtype != np.float32:
        era0[v] = era0[v].astype(np.float32)

# détecter la dim verticale (ERA5 snapshots = souvent 'hybrid')
lev_dim = "hybrid" if "hybrid" in era0.dims else ("level" if "level" in era0.dims else None)
print("lev_dim:", lev_dim)

# --- 1) Warmup compile MINIMAL: 2D only (rapide à compiler) ---
if "surface_pressure" in era0.data_vars:
    t=time.time()
    tmp = xarray_utils.regrid(era0[["surface_pressure"]], regridder)
    _ = tmp["surface_pressure"].values
    print("warmup 2D surface_pressure:", time.time()-t, "s")

# --- 2) Warmup compile 3D: UN SEUL NIVEAU ---
if "temperature" in era0.data_vars and lev_dim is not None and lev_dim in era0["temperature"].dims:
    t=time.time()
    tmp = xarray_utils.regrid(era0[["temperature"]].isel({lev_dim: 0}), regridder)
    _ = tmp["temperature"].values
    print(f"warmup 3D temperature({lev_dim}=0):", time.time()-t, "s")

# --- 3) Regrid 2D vars d'abord ---
out = {}
t_all = time.time()

vars_2d = [v for v in present if lev_dim is None or lev_dim not in era0[v].dims]
vars_3d = [v for v in present if lev_dim is not None and lev_dim in era0[v].dims]

for v in vars_2d:
    t=time.time()
    ds_v = xarray_utils.regrid(era0[[v]], regridder)
    out[v] = ds_v[v]
    _ = out[v].values
    print(f"[regrid 2D] {v}: {time.time()-t:.2f}s  shape={out[v].shape}")

# --- 4) Regrid 3D vars en BLOCS de niveaux (évite graphe géant) ---
block = 8  # ajuste: 4, 8, 16 (8 est un bon départ)
if vars_3d:
    nlev = int(era0.sizes[lev_dim])
    for v in vars_3d:
        parts = []
        print(f"Regridding {v} in level-blocks of {block} (nlev={nlev}) ...")
        for k0 in range(0, nlev, block):
            k1 = min(nlev, k0 + block)
            t=time.time()
            ds_blk = xarray_utils.regrid(era0[[v]].isel({lev_dim: slice(k0, k1)}), regridder)
            da_blk = ds_blk[v]
            # force compute petite tranche pour feedback
            _ = da_blk.isel({lev_dim: 0}).values
            parts.append(da_blk)
            print(f"  levels {k0}:{k1} done in {time.time()-t:.2f}s")
        out[v] = xr.concat(parts, dim=lev_dim)
        print(f"[regrid 3D] {v}: final shape={out[v].shape}")

ds_in = xr.Dataset(out).assign_coords(longitude=lon_da, latitude=lat_da)
print("Done Dinosaur regrid:", ds_in.dims, "TOTAL", time.time()-t_all, "s")

# compat pour la suite
era_on_model = ds_in

missing: []
lev_dim: hybrid


W0225 10:24:22.950734 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


warmup 2D surface_pressure: 0.46732044219970703 s
warmup 3D temperature(hybrid=0): 0.5499129295349121 s
[regrid 2D] surface_pressure: 0.00s  shape=(96, 48)
[regrid 2D] geopotential_at_surface: 0.02s  shape=(96, 48)
Regridding u_component_of_wind in level-blocks of 8 (nlev=137) ...
  levels 0:8 done in 97.25s
  levels 8:16 done in 0.04s
  levels 16:24 done in 0.04s
  levels 24:32 done in 0.04s
  levels 32:40 done in 0.05s
  levels 40:48 done in 0.04s
  levels 48:56 done in 0.04s
  levels 56:64 done in 0.04s
  levels 64:72 done in 0.04s
  levels 72:80 done in 0.05s
  levels 80:88 done in 0.04s
  levels 88:96 done in 0.05s
  levels 96:104 done in 0.04s
  levels 104:112 done in 0.05s
  levels 112:120 done in 0.04s
  levels 120:128 done in 0.05s
  levels 128:136 done in 32.20s
  levels 136:137 done in 0.01s
[regrid 3D] u_component_of_wind: final shape=(137, 96, 48)
Regridding v_component_of_wind in level-blocks of 8 (nlev=137) ...
  levels 0:8 done in 0.04s
  levels 8:16 done in 0.04s
  lev

In [6]:
print(ds_in.dims)
print(list(ds_in.data_vars))

FrozenMappingWarningOnValuesAccess({'longitude': 96, 'latitude': 48, 'hybrid': 137})
['surface_pressure', 'geopotential_at_surface', 'u_component_of_wind', 'v_component_of_wind', 'temperature', 'specific_humidity']


In [8]:
ds = ds_in
print("sizes:", dict(ds.sizes))
print("vars:", list(ds.data_vars))
print("coords:", list(ds.coords))

sizes: {'longitude': 96, 'latitude': 48, 'hybrid': 137}
vars: ['surface_pressure', 'geopotential_at_surface', 'u_component_of_wind', 'v_component_of_wind', 'temperature', 'specific_humidity']
coords: ['time', 'hybrid', 'longitude', 'latitude']


In [10]:
era_on_model = ds_in

In [11]:
print("ds_in sizes:", dict(ds_in.sizes))
print("ds_era sizes:", dict(ds_era.sizes))
print("ds_era has time?", "time" in ds_era.dims)

ds_in sizes: {'longitude': 96, 'latitude': 48, 'hybrid': 137}
ds_era sizes: {'time': 40, 'latitude': 721, 'longitude': 1440, 'hybrid': 137}
ds_era has time? True


In [12]:
import numpy as np
import jax.numpy as jnp

ds = era_on_model  # ERA5 regriddé horizontalement (96x48, hybrid=137)

lat_dim = "latitude"
lon_dim = "longitude"
lev_dim = "hybrid"  # chez toi c’est certain

# --- 137 -> 8 (hack temporaire) ---
nlev_model = int(geom.nodal_shape[0])   # 8
nlev_era = int(ds.sizes[lev_dim])       # 137

idx = np.linspace(0, nlev_era - 1, nlev_model).round().astype(int)
print("Selected hybrid indices:", idx, "from", nlev_era, "->", nlev_model)

ds8 = ds.isel({lev_dim: idx})

# --- helpers: sortir directement (lev, lon, lat) et (lon,lat) ---
def lev_lon_lat(da):
    return jnp.asarray(da.transpose(lev_dim, lon_dim, lat_dim).data)

def lon_lat_2d(da):
    return jnp.asarray(da.transpose(lon_dim, lat_dim).data)

required_3d = ["u_component_of_wind", "v_component_of_wind", "temperature"]
required_2d = ["surface_pressure", "geopotential_at_surface"]
missing = [v for v in required_3d + required_2d if v not in ds8.data_vars]
assert not missing, f"Missing vars in ds8: {missing}"

u_wind = lev_lon_lat(ds8["u_component_of_wind"])
v_wind = lev_lon_lat(ds8["v_component_of_wind"])
temperature = lev_lon_lat(ds8["temperature"])

if "specific_humidity" in ds8.data_vars:
    specific_humidity = lev_lon_lat(ds8["specific_humidity"])
else:
    specific_humidity = None

surface_pressure = lon_lat_2d(ds8["surface_pressure"])
geopotential = lon_lat_2d(ds8["geopotential_at_surface"])

print("u_wind", u_wind.shape)
print("v_wind", v_wind.shape)
print("temperature", temperature.shape)
print("surface_pressure", surface_pressure.shape)
print("geopotential", geopotential.shape)
print("specific_humidity", None if specific_humidity is None else specific_humidity.shape)


Selected hybrid indices: [  0  19  39  58  78  97 117 136] from 137 -> 8
u_wind (8, 96, 48)
v_wind (8, 96, 48)
temperature (8, 96, 48)
surface_pressure (96, 48)
geopotential (96, 48)
specific_humidity (8, 96, 48)


In [13]:
import jcm.physics_interface as pi
import jax.numpy as jnp

P0 = 1e5
normalized_surface_pressure = (surface_pressure / P0)[None, ...]  # (1, lon, lat)

# si q absent: mettre un champ 0 (même shape que temperature)
if specific_humidity is None:
    specific_humidity = jnp.zeros_like(temperature)

initial_physics_state = pi.PhysicsState(
    u_wind=u_wind,
    v_wind=v_wind,
    temperature=temperature,
    specific_humidity=specific_humidity,
    geopotential=geopotential,
    normalized_surface_pressure=normalized_surface_pressure,
)
print("PhysicsState OK.")
print("nsp shape:", initial_physics_state.normalized_surface_pressure.shape)


PhysicsState OK.
nsp shape: (1, 96, 48)


W0225 11:08:23.627147 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:08:23.673857 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


In [14]:
import xarray as xr
import numpy as np

IC_OUT = "/scratch/gmathieu/IC_USED.nc"  # écris direct au bon endroit

data_vars = dict(
    u_wind=(("level","longitude","latitude"), np.asarray(initial_physics_state.u_wind)),
    v_wind=(("level","longitude","latitude"), np.asarray(initial_physics_state.v_wind)),
    temperature=(("level","longitude","latitude"), np.asarray(initial_physics_state.temperature)),
    geopotential=(("longitude","latitude"), np.asarray(initial_physics_state.geopotential)),
    normalized_surface_pressure=(("one","longitude","latitude"),
                                 np.asarray(initial_physics_state.normalized_surface_pressure)),
)

# specific_humidity toujours présent maintenant (zéros si absent)
data_vars["specific_humidity"] = (("level","longitude","latitude"),
                                  np.asarray(initial_physics_state.specific_humidity))

ds_ic = xr.Dataset(
    data_vars=data_vars,
    coords=dict(
        level=np.arange(geom.nodal_shape[0], dtype=np.int32),
        one=np.array([0], dtype=np.int32),
        longitude=np.asarray(lon_deg),
        latitude=np.asarray(lat_deg),
    ),
    attrs=dict(
        source_era5="t0 regridded with Dinosaur (horizontal)",
        era5_vertical_indices=str(idx.tolist()),
        P0_used_Pa=100000.0,
        note="TEMP vertical downselect hybrid->8 (placeholder). Horizontal regrid uses Dinosaur bilinear.",
    )
)

ds_ic.to_netcdf(IC_OUT)
print("WROTE:", IC_OUT)
print("IC vars:", list(ds_ic.data_vars))
print("IC nsp:", ds_ic["normalized_surface_pressure"].shape)


WROTE: /scratch/gmathieu/IC_USED.nc
IC vars: ['u_wind', 'v_wind', 'temperature', 'geopotential', 'normalized_surface_pressure', 'specific_humidity']
IC nsp: (1, 96, 48)


In [15]:
import jax.numpy as jnp
import jcm.physics_interface as pi

class NoPhysics(pi.Physics):
    def compute_tendencies(self, state: pi.PhysicsState, forcing, geometry, date):
        tend = pi.PhysicsTendency(
            u_wind=jnp.zeros_like(state.u_wind),
            v_wind=jnp.zeros_like(state.v_wind),
            temperature=jnp.zeros_like(state.temperature),
            specific_humidity=jnp.zeros_like(state.specific_humidity),
        )
        data = self.get_empty_data(geometry)
        return tend, data


In [16]:
tend0, data0 = NoPhysics().compute_tendencies(initial_physics_state, forcing=None, geometry=geom, date=None)
print("tend u", tend0.u_wind.shape, tend0.u_wind.dtype, float(tend0.u_wind.max()))
print("data", data0)


tend u (8, 96, 48) float32 0.0
data None


W0225 11:09:22.173847 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:09:22.225194 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


In [17]:
from jcm.model import Model

model_dry = Model(geometry=geom, physics=NoPhysics())
print("model_dry built:", model_dry)


model_dry built: <jcm.model.Model object at 0x1499451a80d0>


In [18]:
import numpy as np
print("T0 min/max:", float(np.asarray(initial_physics_state.temperature).min()),
                 float(np.asarray(initial_physics_state.temperature).max()))


T0 min/max: 177.02076721191406 308.027099609375


In [19]:
from jcm.physics.speedy.speedy_physics import SpeedyPhysics

physics_full = SpeedyPhysics()   # paramètres par défaut (ok pour commencer)
model_full = Model(geometry=geom, physics=physics_full)

print("model_full built:", model_full)



model_full built: <jcm.model.Model object at 0x1499451ab350>


In [20]:
import os, jcm
from jcm.forcing import ForcingData

JCM_ROOT = os.path.dirname(jcm.__file__)
FORCING = os.path.join(JCM_ROOT, "data/bc/t30/clim/forcing.nc")

forcing = ForcingData.from_file(FORCING, target_resolution=31)
print("forcing:", type(forcing))


/home/gmathieu/src/jax-gcm/jcm/data/bc/interpolate.py:29: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge([daily_time_vars, ds_monthly[non_time_vars]])
W0225 11:14:59.528155 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:14:59.558158 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:14:59.610760 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:14:59.659448 3244815 sol_gpu_cost_model.cc:102] No SoL 

forcing: <class 'jcm.forcing.ForcingData'>


In [21]:
SAVE_H = 3.0
TOTAL_H = 6.0

pred_dry_test = model_dry.run(
    initial_state=initial_physics_state,
    forcing=forcing,
    save_interval=SAVE_H,
    total_time=TOTAL_H,
)
print("dry TEST run OK")

pred_full_test = model_full.run(
    initial_state=initial_physics_state,
    forcing=forcing,
    save_interval=SAVE_H,
    total_time=TOTAL_H,
)
print("full TEST run OK")

ds_full_test = pred_full_test.to_xarray()
ds_dry_test  = pred_dry_test.to_xarray()
print("TEST ds_full dims:", ds_full_test.dims)
print("TEST ds_dry  dims:", ds_dry_test.dims)

W0225 11:18:38.510203 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.524402 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.548357 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.665666 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.710708 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.757064 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:38.802389 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. 

dry TEST run OK


W0225 11:18:48.014889 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:48.031131 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:48.048134 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:48.061962 3244910 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:18:53.301196 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


full TEST run OK


W0225 11:19:11.268894 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.338536 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.390433 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.420138 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.475173 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.526002 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.
W0225 11:19:11.555257 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. 

TEST ds_full dims: FrozenMappingWarningOnValuesAccess({'time': 2, 'lon': 96, 'lat': 48, 'level': 8})
TEST ds_dry  dims: FrozenMappingWarningOnValuesAccess({'time': 2, 'level': 8, 'lon': 96, 'lat': 48})


W0225 11:19:11.677196 3244815 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H100 80GB HBM3 MIG 2g.20gb. Using default config.


In [28]:
SAVE = 0.125   # days = 3h
TOTAL = 5.0    # days

OUT_FULL = "/scratch/gmathieu/pred_dino_full_5days_3h.nc"
OUT_DRY  = "/scratch/gmathieu/pred_dino_only_5days_3h.nc"

pred_dry = model_dry.run(
    initial_state=initial_physics_state,
    forcing=forcing,
    save_interval=SAVE_H,
    total_time=TOTAL_H,
)
print("dry FINAL run OK")

pred_full = model_full.run(
    initial_state=initial_physics_state,
    forcing=forcing,
    save_interval=SAVE_H,
    total_time=TOTAL_H,
)
print("full FINAL run OK")

ds_full = pred_full.to_xarray()
ds_dry  = pred_dry.to_xarray()

ds_full.to_netcdf(OUT_FULL)
ds_dry.to_netcdf(OUT_DRY)
print("WROTE:", OUT_FULL)
print("WROTE:", OUT_DRY)

print("FINAL ds_full dims:", ds_full.dims)
print("FINAL ds_dry  dims:", ds_dry.dims)

dry FINAL run OK
full FINAL run OK
WROTE: /scratch/gmathieu/pred_dino_full_5days_3h.nc
WROTE: /scratch/gmathieu/pred_dino_only_5days_3h.nc
FINAL ds_full dims: FrozenMappingWarningOnValuesAccess({'time': 40, 'lon': 96, 'lat': 48, 'level': 8})
FINAL ds_dry  dims: FrozenMappingWarningOnValuesAccess({'time': 40, 'level': 8, 'lon': 96, 'lat': 48})


In [30]:
import os, glob
import xarray as xr
import pandas as pd

ERA_DIR = "/scratch/gmathieu/era5_snapshots_5days_3h_1990"
FULL_PATH = "/scratch/gmathieu/pred_dino_full_5days_3h.nc"
DRY_PATH  = "/scratch/gmathieu/pred_dino_only_5days_3h.nc"

# ERA5 times from filenames
era_files = sorted(glob.glob(os.path.join(ERA_DIR, "era5_*.nc")))
assert len(era_files) > 0, f"No ERA5 files found in {ERA_DIR}"

era_times = pd.to_datetime(
    [os.path.basename(f).split("_")[1].split(".")[0] for f in era_files],
    format="%Y%m%dT%H",
)
print("ERA N:", len(era_times), "first/last:", era_times[0], era_times[-1])

def overwrite_time(path):
    ds = xr.open_dataset(path)
    assert "time" in ds.dims, "no time dim"
    assert ds.sizes["time"] == len(era_times), f"time mismatch: {ds.sizes['time']} vs {len(era_times)}"

    ds2 = ds.assign_coords(time=("time", era_times.values))

    tmp = path + ".tmp"
    ds2.to_netcdf(tmp)
    os.replace(tmp, path)

    ds.close()
    print("✅ time overwritten:", path)

overwrite_time(FULL_PATH)
overwrite_time(DRY_PATH)

ERA N: 40 first/last: 1990-05-01 00:00:00 1990-05-05 21:00:00
✅ time overwritten: /scratch/gmathieu/pred_dino_full_5days_3h.nc
✅ time overwritten: /scratch/gmathieu/pred_dino_only_5days_3h.nc


In [31]:
import os, glob
import xarray as xr
import pandas as pd

ERA_DIR = "/scratch/gmathieu/era5_snapshots_5days_3h_1990"
paths = {
    "FULL": "/scratch/gmathieu/pred_dino_full_5days_3h.nc",
    "DRY" : "/scratch/gmathieu/pred_dino_only_5days_3h.nc",
}

# --- ERA5 times from filenames (source of truth) ---
era_files = sorted(glob.glob(os.path.join(ERA_DIR, "era5_*.nc")))
assert len(era_files) > 0, f"No ERA5 files found in {ERA_DIR}"

t_e = pd.to_datetime(
    [os.path.basename(f).split("_")[1].split(".")[0] for f in era_files],
    format="%Y%m%dT%H",
)

print("ERA5  first/last/N =", t_e[0], t_e[-1], len(t_e))
print("ERA5  Δt counts:\n", (t_e[1:] - t_e[:-1]).to_series().value_counts())

for name, path in paths.items():
    if not os.path.exists(path):
        print(f"⚠️ {name}: not found: {path}")
        continue

    ds = xr.open_dataset(path)
    assert "time" in ds.dims, f"{name}: no time dim in {path}"
    t_m = pd.to_datetime(ds["time"].values)

    print(f"\n{name} first/last/N =", t_m[0], t_m[-1], len(t_m))
    print(f"{name} Δt counts:\n", (t_m[1:] - t_m[:-1]).to_series().value_counts())

    if len(t_m) != len(t_e):
        print(f"❌ {name} N mismatch: model={len(t_m)} ERA={len(t_e)}")
        continue

    diff = (t_m - t_e)
    print(f"{name} max |Δt| =", pd.to_timedelta(diff).map(abs).max())
    if not (t_m == t_e).all():
        bad = (t_m != t_e)
        bad_idx = list(pd.Series(range(len(t_m)))[bad][:10])
        print(f"❌ {name} mismatch at indices (first 10):", bad_idx)
    else:
        print(f"✅ {name} times exactly match ERA5 snapshot times")

ERA5  first/last/N = 1990-05-01 00:00:00 1990-05-05 21:00:00 40
ERA5  Δt counts:
 0 days 03:00:00    39
Name: count, dtype: int64

FULL first/last/N = 1990-05-01 00:00:00 1990-05-05 21:00:00 40
FULL Δt counts:
 0 days 03:00:00    39
Name: count, dtype: int64
FULL max |Δt| = 0 days 00:00:00
✅ FULL times exactly match ERA5 snapshot times

DRY first/last/N = 1990-05-01 00:00:00 1990-05-05 21:00:00 40
DRY Δt counts:
 0 days 03:00:00    39
Name: count, dtype: int64
DRY max |Δt| = 0 days 00:00:00
✅ DRY times exactly match ERA5 snapshot times


In [32]:
import xarray as xr
import numpy as np

ds_ic = xr.Dataset(
    data_vars=dict(
        u_wind=(("level","longitude","latitude"), np.asarray(initial_physics_state.u_wind)),
        v_wind=(("level","longitude","latitude"), np.asarray(initial_physics_state.v_wind)),
        temperature=(("level","longitude","latitude"), np.asarray(initial_physics_state.temperature)),
        specific_humidity=(("level","longitude","latitude"), np.asarray(initial_physics_state.specific_humidity)),
        geopotential=(("longitude","latitude"), np.asarray(initial_physics_state.geopotential)),
        normalized_surface_pressure=(("one","longitude","latitude"), np.asarray(initial_physics_state.normalized_surface_pressure)),
    ),
    coords=dict(
        level=np.arange(geom.nodal_shape[0], dtype=np.int32),
        one=np.array([0], dtype=np.int32),
        longitude=lon_deg,
        latitude=lat_deg,
    ),
    attrs=dict(
        source_era5=ERA5_PATH,
        era5_vertical_indices=str(idx.tolist()),
        P0_used_Pa=100000.0,
        note="IC used for both dino-only (NoPhysics) and dino-full (SpeedyPhysics). Shapes follow model order.",
    )
)
ds_ic.to_netcdf("/scratch/gmathieu/IC_USED.nc")
print("WROTE: /scratch/gmathieu/IC_USED.nc")


WROTE: /scratch/gmathieu/IC_USED.nc


In [33]:
import xarray as xr
import numpy as np

full_path = "/scratch/gmathieu/pred_dino_full_5days_3h.nc"
dry_path  = "/scratch/gmathieu/pred_dino_only_5days_3h.nc"

ds_full = xr.open_dataset(full_path)
ds_dry  = xr.open_dataset(dry_path)

print("FULL dims:", dict(ds_full.sizes))
print("DRY  dims:", dict(ds_dry.sizes))

print("\nFULL vs DRY at t=0 (level=0) maxabs:")
for v in ["temperature", "u_wind", "v_wind"]:
    if v in ds_full and v in ds_dry:
        d = (ds_full[v].isel(time=0, level=0) - ds_dry[v].isel(time=0, level=0)).values
        print(v, float(np.nanmax(np.abs(d))))

FULL dims: {'time': 40, 'lon': 96, 'lat': 48, 'level': 8}
DRY  dims: {'time': 40, 'level': 8, 'lon': 96, 'lat': 48}

FULL vs DRY at t=0 (level=0) maxabs:
temperature 0.0
u_wind 0.0
v_wind 0.0


In [2]:
import xarray as xr
ds_full = xr.open_dataset("/scratch/gmathieu/pred_dino_full_5days_3h.nc")
ds_dry  = xr.open_dataset("/scratch/gmathieu/pred_dino_only_5days_3h.nc")

print("FULL dims:", dict(ds_full.sizes))
print("FULL vars:", list(ds_full.data_vars))
print("DRY  vars:", list(ds_dry.data_vars))

FULL dims: {'time': 40, 'lon': 96, 'lat': 48, 'level': 8}
FULL vars: ['surface_flux.hfluxn.0', 'convection.precnv', 'shortwave_rad.ftop', 'surface_flux.shf.2', 'surface_flux.rlus.0', 'surface_flux.vstr.0', 'shortwave_rad.rsds', 'condensation.precls', 'surface_flux.tsfc', 'surface_flux.vstr.1', 'shortwave_rad.ozupp', 'mod_radcon.albsfc', 'convection.cbmf', 'mod_radcon.tau2.1', 'convection.iptop', 'surface_flux.rlds', 'condensation.dtlsc', 'surface_flux.vstr.2', 'mod_radcon.flux.1', 'specific_humidity', 'longwave_rad.dfabs', 'shortwave_rad.gse', 'shortwave_rad.dfabs', 'surface_flux.tskin', 'humidity.rh', 'mod_radcon.tau2.0', 'convection.se', 'mod_radcon.tau2.2', 'surface_flux.shf.0', 'mod_radcon.flux.3', 'mod_radcon.flux.2', 'shortwave_rad.ozone', 'surface_flux.ustr.1', 'surface_flux.rlns', 'mod_radcon.snowc', 'surface_flux.u0', 'date.tyear', 'surface_flux.rlus.2', 'shortwave_rad.icltop', 'shortwave_rad.qcloud', 'mod_radcon.stratc.0', 'surface_flux.hfluxn.1', 'temperature', 'surface_flux

In [3]:
import xarray as xr
ds=xr.open_dataset("/scratch/gmathieu/era5_snapshots_5days_3h_1990/era5_19900501T00.nc")
print(list(ds.data_vars))
print(dict(ds.sizes))

['surface_pressure', 'geopotential_at_surface', 'temperature', 'u_component_of_wind', 'v_component_of_wind', 'specific_humidity']
{'latitude': 721, 'longitude': 1440, 'hybrid': 137}
